# SVCJ identification diagnostics

This notebook inspects fitted SVCJ parameter paths without changing calibration, pricing, or hedging conventions. It reports Feller-gap behavior, Brownian-correlation boundary clustering, jump-parameter near-bound behavior, moment-domain margins, the empirical distribution of the unbounded jump loading `rho_j`, and parameter correlations.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / 'notebooks' / '_lib').exists():
    PROJECT_ROOT = ROOT
    NOTEBOOK_DIR = ROOT / 'notebooks'
elif (ROOT / '_lib').exists():
    NOTEBOOK_DIR = ROOT
    PROJECT_ROOT = ROOT.parent
else:
    raise FileNotFoundError('Could not locate notebooks/_lib')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
from _lib import chapter3_final, robustness

REMEDIATION_DIR = PROJECT_ROOT / 'excel files' / 'remediation'
OUTPUT_DIR = NOTEBOOK_DIR / 'chapter3_outputs' / 'remediation' / 'identification'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REGULARISATIONS = (0, 1)


In [ ]:
available = {}
for reg in REGULARISATIONS:
    path = REMEDIATION_DIR / f'calibration_results_reg_{reg}.xlsx'
    if path.exists():
        available[reg] = path
    else:
        print(f'Pending Task 13: {path.name} does not exist yet')
available


In [ ]:
identification_tables = []
for reg, path in available.items():
    _, _, svcj_params, _ = chapter3_final.load_baseline_workbook(path, evaluation_split='test')
    ident = robustness.parameter_identification_summary(svcj_params)
    ident['lambda_TS'] = reg
    identification_tables.append(ident)
    for currency in ('BTC', 'ETH'):
        corr = robustness.parameter_correlation_matrix(svcj_params, currency=currency)
        corr.to_csv(OUTPUT_DIR / f'svcj_parameter_correlations_reg_{reg}_{currency}.csv')

identification_summary = pd.concat(identification_tables, ignore_index=True) if identification_tables else pd.DataFrame()
if not identification_summary.empty:
    identification_summary.to_csv(OUTPUT_DIR / 'svcj_identification_summary.csv', index=False)
identification_summary
